# VishGym alternating QLoRA/GRPO training
This notebook is designed to be opened and operated through the Google Colab MCP bridge. It trains only on reviewed, synthetic artifacts and does not publish candidates automatically.

In [ ]:
!git clone https://github.com/REPLACE_WITH_OWNER/VishGym.git
%cd VishGym
!pip install -e '.[training,audio]'

In [ ]:
from vishgym.arena.runner import run_local_episode
state, verdict = run_local_episode(seed=7)
assert verdict.terminal_outcome in {'safe_defense', 'simulated_compromise', 'false_block', 'inconclusive'}
print(verdict.model_dump())

In [ ]:
import json
from pathlib import Path
from vishgym.training.synthetic_data import build_warm_start_examples

examples = build_warm_start_examples(seeds=[7, 11])
Path('artifacts/training').mkdir(parents=True, exist_ok=True)
Path('artifacts/training/warm_start.synthetic.jsonl').write_text(
    ''.join(json.dumps(example.model_dump()) + '\n' for example in examples), encoding='utf-8'
)
print(f'Wrote {len(examples)} reviewed-synthetic examples.')

In [ ]:
from vishgym.training.evaluation import EvaluationReport, promotion_decision, write_review_manifest

# After each reviewed Red/Blue GRPO round, replace these measured values from held-out rollouts.
report = EvaluationReport(
    dataset_revision='warm-start.synthetic.v1', adapter_revision='blue-round-1-candidate',
    true_positive=80, false_positive=5, false_negative=5,
    legitimate_cases=100, legitimate_false_blocks=8, boundary_violations=0,
    valid_tool_calls=99, total_tool_calls=100, seed_set=[7, 11],
)
decision = promotion_decision(report)
manifest = write_review_manifest('artifacts/training/blue-round-1.review.json', role='blue', candidate_adapter='LOCAL_REVIEWED_PATH', report=report)
print(decision, manifest)

## Training contract
## Alternating QLoRA/GRPO contract
Run three rounds with a shared 4-bit Gemma base: (1) train Red against frozen Blue, (2) evaluate held-out personas/timbres/scenarios and review the candidate, then (3) train Blue against that frozen Red. Retain historical opponent checkpoints. Persist all seeds, perturbations, metrics, and candidate manifests. Human approval is required before publishing any Hugging Face model artifact. The Colab MCP operator should execute these cells and record the run links in the reviewer log.